# Stochastic Anchor Growing — Experiment Notebook

This notebook reproduces the experiments from the project report
*"Stochastic Anchor Growing: a Sparse-View Generalization of Scaffold-GS"*
(Guillaume Bernard, MVA 2025-2026).

## Setup

- **Environment**: Google Colab (Tesla T4 GPU)
- **Storage**: Google Drive, mounted at `/content/drive`
- **Scaffold-GS repo**: `MyDrive/Scaffold-GS-probgrow/`
  (standard Scaffold-GS with `scene/gaussian_model.py` replaced by the
  modified version provided in this repository)
- **Datasets**: `MyDrive/datasets/{scene}_{sparsity}/`
  (DrJohnson and South-Building, subsampled to 20% and 50% of original views;
  COLMAP re-run from scratch on each subset)
- **ONNX export tool**: `MyDrive/visionary/`
  (from the visionary-laboratory repository)

## Stochastic Growing Parameters

The modified `gaussian_model.py` exposes three hyperparameters at the top
of `GaussianModel.__init__`:
```python
self.prob_grow = True              # set to False for standard baseline
self.prob_grow_low_mul = 0.1       # alpha: lower gradient threshold multiplier
self.prob_grow_obs_bonus = 1.5     # beta:  under-observation bonus
self.prob_grow_min_obs_ratio = 0.75 # gamma: minimum observation gate
```

Change these values directly in `gaussian_model.py` before launching a run
to explore different parameter configurations.

## Visualizing Results

After each training run, the notebook exports the reconstruction to `.onnx`
format. To visualize it, open the **Visionary viewer** and drag-and-drop
the `.onnx` file:

👉 https://visionary-laboratory.github.io/visionary/index_visionary.html

# Loading Scaffold-GS:

In [ ]:
!nvidia-smi

Wed Mar 18 21:59:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

/content/drive/MyDrive/Scaffold-GS-probgrow


In [ ]:
!apt-get update -y
!apt-get install -y build-essential ninja-build

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,452 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,880 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,935 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/un

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.7 MB/s eta 0:00:00


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)

torch: 2.10.0+cu128
cuda available: True
torch cuda: 12.8


In [ ]:
!pip install plyfile tqdm einops wandb lpips laspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 9.0 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow/submodules/diff-gaussian-rasterization
!pip install -v .

/content/drive/MyDrive/Scaffold-GS-probgrow/submodules/diff-gaussian-rasterization
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing /content/drive/MyDrive/Scaffold-GS-probgrow/submodules/diff-gaussian-rasterization
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-hjw0ojkh/diff_gaussian_rasterization.egg-info
  writing /tmp/pip-pip-egg-info-hjw0ojkh/diff_gaussian_rasterization.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-hjw0ojkh/diff_gaussian_rasterization.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-hjw0ojkh/diff_gaussian_rasterization.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-hjw0ojkh/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-hjw0ojkh/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  adding license file 'LICENSE.md'
  writing manifest file '/tmp/pip-pip-

In [ ]:
from pathlib import Path

f = Path("/content/drive/MyDrive/Scaffold-GS-probgrow/submodules/simple-knn/simple_knn.cu")
text = f.read_text()

if "#include <float.h>" not in text and "#include <cfloat>" not in text:
    lines = text.splitlines()
    insert_at = 0
    while insert_at < len(lines) and lines[insert_at].startswith("#include"):
        insert_at += 1
    lines.insert(insert_at, "#include <float.h>")
    f.write_text("\n".join(lines) + "\n")
    print("Patched:", f)
else:
    print("Header already present:", f)

print("First 20 lines:\n")
print("\n".join(f.read_text().splitlines()[:20]))

Header already present: /content/drive/MyDrive/Scaffold-GS-probgrow/submodules/simple-knn/simple_knn.cu
First 20 lines:

#include <float.h>
/*
 * Copyright (C) 2023, Inria
 * GRAPHDECO research group, https://team.inria.fr/graphdeco
 * All rights reserved.
 *
 * This software is free for non-commercial, research and evaluation use 
 * under the terms of the LICENSE.md file.
 *
 * For inquiries contact  george.drettakis@inria.fr
 */

#define BOX_SIZE 1024

#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include "simple_knn.h"
#include <cub/cub.cuh>
#include <cub/device/device_radix_sort.cuh>
#include <vector>


In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow/submodules/simple-knn
!pip uninstall -y simple_knn
!MAX_JOBS=1 pip install -v . --no-build-isolation

/content/drive/MyDrive/Scaffold-GS-probgrow/submodules/simple-knn
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing /content/drive/MyDrive/Scaffold-GS-probgrow/submodules/simple-knn
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info
  writing /tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-xi6jw352/simple_knn.egg-info/SOURCES.txt'
  Preparing metadata (setup.py) ... done
  Running command python setup.py bdist_wheel
  running bdist_wheel
  running b

In [ ]:
!pip install colorama

In [ ]:
!cp /content/drive/MyDrive/Scaffold-GS-probgrow/scene/gaussian_model.py /content/drive/MyDrive/Scaffold-GS-probgrow/scene/gaussian_model.py.bak

In [ ]:
from pathlib import Path

path = Path("/content/drive/MyDrive/Scaffold-GS-probgrow/scene/gaussian_model.py")
text = path.read_text()

old = "from torch_scatter import scatter_max"
new = """import torch

def scatter_max(src, index, dim=0):
    if dim != 0:
        raise NotImplementedError("Temporary replacement only supports dim=0")
    if index.numel() == 0:
        out = torch.empty((0,), device=src.device, dtype=src.dtype)
        arg = torch.empty((0,), device=index.device, dtype=index.dtype)
        return out, arg
    out_size = int(index.max().item()) + 1
    out = torch.full((out_size,), -torch.inf, device=src.device, dtype=src.dtype)
    out.scatter_reduce_(0, index, src, reduce="amax", include_self=True)
    arg = None
    return out, arg
"""

if old in text:
    text = text.replace(old, new, 1)
    path.write_text(text)
    print("Patched successfully.")
else:
    print("Did not find the original import line.")

Did not find the original import line.


In [ ]:
!grep -n "scatter_max" /content/drive/MyDrive/Scaffold-GS-probgrow/scene/gaussian_model.py | head -n 20

17:def scatter_max(src, index, dim=0):
711:                new_feat = scatter_max(new_feat, inverse_indices.unsqueeze(1).expand(-1, new_feat.size(1)), dim=0)[0][remove_duplicates]


In [ ]:
!pip install jaxtyping

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow
import torch
print("CUDA available:", torch.cuda.is_available())

import gaussian_renderer
print("gaussian_renderer import OK")

/content/drive/MyDrive/Scaffold-GS-probgrow
CUDA available: True
gaussian_renderer import OK


In [ ]:
!python train.py --help

0
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:02<00:00, 232MB/s]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
2026-03-18 22:05:50.002611: E external/local_xla/xla/strea

In [ ]:
!pip install lpips

# DATASET :

In [ ]:
!mkdir -p /content/datasets

In [ ]:
!cp -r /content/drive/MyDrive/datasets/drjohnson /content/datasets/
!ls /content/datasets/drjohnson
!ls /content/datasets/drjohnson/sparse/0

images	sparse
cameras.bin  images.bin  points3D.bin  project.ini


In [ ]:
!mkdir -p /content/scaffold_runs

# EXPORT ONNX :

In [ ]:
%cd /content
!git clone https://github.com/Visionary-Laboratory/visionary.git

/content
Cloning into 'visionary'...
remote: Enumerating objects: 2299, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 2299 (delta 41), reused 35 (delta 35), pack-reused 2250 (from 2)
Receiving objects: 100% (2299/2299), 25.55 MiB | 48.99 MiB/s, done.
Resolving deltas: 100% (960/960), done.


In [ ]:
%cd /content/visionary/onnx-export/ONNXExample-scaffold

/content/visionary/onnx-export/ONNXExample-scaffold


In [ ]:
!pip install onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 17.4 MB/s eta 0:00:00


In [ ]:
file_path = "/content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py"

with open(file_path, "r", encoding="utf-8") as f:
    txt = f.read()

txt = txt.replace(
    'assert init.data_type != TensorProto.FLOAT16, f"initializer {init.name} is fp16"',
    'pass  # fp16 allowed for Scaffold-GS export'
)

with open(file_path, "w", encoding="utf-8") as f:
    f.write(txt)

print("Patched final fp16 assertion only.")

Patched final fp16 assertion only.


# EXPERIENCES :

In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

!python train.py \
    -s /content/datasets/drjohnson \
    -m /content/scaffold_runs/drjohnson_test_probgrow \
    --iterations 5000 \
    --appearance_dim 0

/content/drive/MyDrive/Scaffold-GS-probgrow
0
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
2026-03-09 14:44:33.649219: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when o

In [ ]:
!python /content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py \
  --ply /content/scaffold_runs/drjohnson_test_probgrow/point_cloud/iteration_5000/point_cloud.ply \
  --cfg_args /content/scaffold_runs/drjohnson_test_probgrow/cfg_args \
  --out /content/scaffold_runs/drjohnson_test_probgrow/gaussians3d_scaffold_5k_probgrow.onnx \
  --opset 18

读取配置文件: /content/scaffold_runs/drjohnson_test_probgrow/cfg_args
已加载配置参数: feat_dim=32, appearance_dim=0, n_offsets=10, use_feat_bank=False, add_opacity_dist=False, add_cov_dist=False, add_color_dist=False
使用 MLP 路径: /content/scaffold_runs/drjohnson_test_probgrow/point_cloud/iteration_5000
/content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py:575: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`...
------camera_center
FakeTensor(..., size=(1, 3))
---
torch.Size([2330580, 3])
---
torch.Size([2330580, 3])
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
/usr/lib/python3.12/copyreg.py:99: FutureWa

# CREATION DU DATASET SPARSE-VIEW 20 pourcent

In [ ]:
import shutil
from pathlib import Path
import random

src = Path("/content/datasets/drjohnson/images")
dst_root = Path("/content/datasets/drjohnson_20")

dst_images = dst_root / "images"
dst_images.mkdir(parents=True, exist_ok=True)

imgs = sorted(list(src.glob("*.jpg")))

ratio = 0.2
k = int(len(imgs) * ratio)

selected = sorted(random.sample(imgs, k))

for p in selected:
    shutil.copy2(p, dst_images / p.name)

print("Images sélectionnées:", len(selected), "/", len(imgs))

Images sélectionnées: 52 / 263


In [ ]:
!apt-get update
!apt-get install -y colmap

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
!rm -f /content/datasets/drjohnson_20/database.db
!rm -rf /content/datasets/drjohnson_20/sparse
!mkdir -p /content/datasets/drjohnson_20/sparse

In [ ]:
!QT_QPA_PLATFORM=offscreen colmap feature_extractor \
    --database_path /content/datasets/drjohnson_20/database.db \
    --image_path /content/datasets/drjohnson_20/images \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0


Feature extraction

Processed file [1/52]
  Name:            IMG_6319.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        4958
Processed file [2/52]
  Name:            IMG_6314.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        5982
Processed file [3/52]
  Name:            IMG_6323.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        6177
Processed file [4/52]
  Name:            IMG_6302.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        9056
Processed file [5/52]
  Name:            IMG_6296.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        11222
Processed file [6/52]
  Name:            IMG_6312.jpg
  Dimensions:      1332 x 876
  Camera:          

In [ ]:
!QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher \
    --database_path /content/datasets/drjohnson_20/database.db \
    --SiftMatching.use_gpu 0


Exhaustive feature matching

Matching block [1/2, 1/2] in 227.001s
Matching block [1/2, 2/2] in 1.347s
Matching block [2/2, 1/2] in 17.886s
Matching block [2/2, 2/2] in 0.896s
Elapsed time: 4.119 [minutes]


In [ ]:
!QT_QPA_PLATFORM=offscreen colmap mapper \
    --database_path /content/datasets/drjohnson_20/database.db \
    --image_path /content/datasets/drjohnson_20/images \
    --output_path /content/datasets/drjohnson_20/sparse

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  96  4.095420e+02    7.49e-03    3.02e+02   3.66e+00   5.99e-01  5.86e+06        1    5.98e-03    5.86e-01
  97  4.095345e+02    7.41e-03    3.01e+02   3.65e+00   5.99e-01  5.90e+06        1    5.79e-03    5.92e-01
  98  4.095272e+02    7.34e-03    2.99e+02   3.65e+00   5.99e-01  5.95e+06        1    5.69e-03    5.98e-01
  99  4.095199e+02    7.27e-03    2.98e+02   3.65e+00   5.99e-01  5.99e+06        1    5.51e-03    6.03e-01
 100  4.095127e+02    7.20e-03    2.97e+02   3.64e+00   5.99e-01  6.04e+06        1    5.60e-03    6.09e-01


Bundle adjustment report
------------------------
    Residuals : 5640
   Parameters : 4135
   Iterations : 101
         Time : 0.609018 [s]
 Initial cost : 0.285891 [px]
   Final cost : 0.26946 [px]
  Termination : No convergence

  => Completed observations: 1
  => Merged observations: 0
  => Filtered observations: 4
  => Changed observations: 0.001773

Global bundle adjustme

In [ ]:
!find /content/datasets/drjohnson_20/sparse -maxdepth 2 -type f

/content/datasets/drjohnson_20/sparse/0/points3D.bin
/content/datasets/drjohnson_20/sparse/0/images.bin
/content/datasets/drjohnson_20/sparse/0/cameras.bin
/content/datasets/drjohnson_20/sparse/0/project.ini


In [ ]:
!ls /content/datasets/drjohnson_20/images | wc -l

52


# CREATION DU DATASET SPARSE VIEW 50 pourcent

In [ ]:
import shutil
from pathlib import Path
import random

src = Path("/content/datasets/drjohnson/images")
dst_root = Path("/content/datasets/drjohnson_50")

dst_images = dst_root / "images"
dst_images.mkdir(parents=True, exist_ok=True)

imgs = sorted(list(src.glob("*.jpg")))

ratio = 0.5
k = int(len(imgs) * ratio)

selected = sorted(random.sample(imgs, k))

for p in selected:
    shutil.copy2(p, dst_images / p.name)

print("Images sélectionnées:", len(selected), "/", len(imgs))

Images sélectionnées: 131 / 263


In [ ]:
!apt-get update
!apt-get install -y colmap

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
!rm -f /content/datasets/drjohnson_50/database.db
!rm -rf /content/datasets/drjohnson_50/sparse
!mkdir -p /content/datasets/drjohnson_50/sparse

In [ ]:
!QT_QPA_PLATFORM=offscreen colmap feature_extractor \
    --database_path /content/datasets/drjohnson_50/database.db \
    --image_path /content/datasets/drjohnson_50/images \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0


Feature extraction

Processed file [1/131]
  Name:            IMG_6311.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        4402
Processed file [2/131]
  Name:            IMG_6293.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        4535
Processed file [3/131]
  Name:            IMG_6317.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        5593
Processed file [4/131]
  Name:            IMG_6310.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        4888
Processed file [5/131]
  Name:            IMG_6296.jpg
  Dimensions:      1332 x 876
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1598.40px
  Features:        11222
Processed file [6/131]
  Name:            IMG_6318.jpg
  Dimensions:      1332 x 876
  Camera:    

In [ ]:
!QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher \
    --database_path /content/datasets/drjohnson_50/database.db \
    --SiftMatching.use_gpu 0


Exhaustive feature matching

Matching block [1/3, 1/3] in 161.365s
Matching block [1/3, 2/3] in 176.013s
Matching block [1/3, 3/3] in 78.934s
Matching block [2/3, 1/3] in 148.134s
Matching block [2/3, 2/3] in 153.989s
Matching block [2/3, 3/3] in 68.460s
Matching block [3/3, 1/3] in 150.659s
Matching block [3/3, 2/3] in 158.524s
Matching block [3/3, 3/3] in 76.426s
Elapsed time: 19.542 [minutes]


In [ ]:
!QT_QPA_PLATFORM=offscreen colmap mapper \
    --database_path /content/datasets/drjohnson_50/database.db \
    --image_path /content/datasets/drjohnson_50/images \
    --output_path /content/datasets/drjohnson_50/sparse

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
         Time : 0.026376 [s]
 Initial cost : 0.393535 [px]
   Final cost : 0.391139 [px]
  Termination : Convergence

  => Merged observations: 0
  => Completed observations: 0
  => Filtered observations: 0
  => Changed observations: 0.000000

Registering image #92 (53)

  => Image sees 177 / 746 points

Pose refinement report
----------------------
    Residuals : 334
   Parameters : 6
   Iterations : 11
         Time : 0.00308704 [s]
 Initial cost : 0.562624 [px]
   Final cost : 0.543847 [px]
  Termination : Convergence

  => Continued observations: 162
  => Added observations: 125

Bundle adjustment report
------------------------
    Residuals : 6382
   Parameters : 1121
   Iterations : 24
         Time : 0.14188 [s]
 Initial cost : 0.389664 [px]
   Final cost : 0.368899 [px]
  Termination : Convergence

  => Merged observations: 44
  => Completed observations: 12
  => Filtered observations: 16
  => Chang

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/datasets/drjohnson/images")
dst_root = Path("/content/datasets/drjohnson_50")
dst_images = dst_root / "images"

if dst_root.exists():
    shutil.rmtree(dst_root)

dst_images.mkdir(parents=True, exist_ok=True)

imgs = sorted(src.glob("*.jpg"))
selected = imgs[::2]   # 50%

for p in selected:
    shutil.copy2(p, dst_images / p.name)

print("Images sélectionnées:", len(selected), "/", len(imgs))

Images sélectionnées: 132 / 263


In [ ]:
!rm -f /content/datasets/drjohnson_50/database.db
!rm -rf /content/datasets/drjohnson_50/sparse
!mkdir -p /content/datasets/drjohnson_50/sparse

!QT_QPA_PLATFORM=offscreen colmap feature_extractor \
    --database_path /content/datasets/drjohnson_50/database.db \
    --image_path /content/datasets/drjohnson_50/images \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0

!QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher \
    --database_path /content/datasets/drjohnson_50/database.db \
    --SiftMatching.use_gpu 0

!QT_QPA_PLATFORM=offscreen colmap mapper \
    --database_path /content/datasets/drjohnson_50/database.db \
    --image_path /content/datasets/drjohnson_50/images \
    --output_path /content/datasets/drjohnson_50/sparse

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  10  1.181021e+04    1.25e-06    5.21e-02   6.56e-03   1.04e+00  5.90e+08        1    1.15e-01    2.09e+00


Bundle adjustment report
------------------------
    Residuals : 76888
   Parameters : 34318
   Iterations : 11
         Time : 2.09445 [s]
 Initial cost : 0.403387 [px]
   Final cost : 0.391922 [px]
  Termination : Convergence

  => Completed observations: 4
  => Merged observations: 0
  => Filtered observations: 1
  => Changed observations: 0.000130
  => Filtered images: 0

Registering image #40 (69)

  => Image sees 213 / 2360 points

Pose refinement report
----------------------
    Residuals : 410
   Parameters : 6
   Iterations : 10
         Time : 0.00349212 [s]
 Initial cost : 0.579375 [px]
   Final cost : 0.575386 [px]
  Termination : Convergence

  => Continued observations: 203
  => Added observations: 415

Bundle adjustment report
------------------------
    Residuals : 5946
   Parameter

# EXPERIENCE 20% :

In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

!python train.py \
    -s /content/datasets/drjohnson_20 \
    -m /content/scaffold_runs/drjohnson20_prob_20k \
    --iterations 20000 \
    --appearance_dim 0

/content/drive/MyDrive/Scaffold-GS-probgrow
0
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
2026-03-12 11:49:19.898223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when o

In [ ]:
!python /content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py \
  --ply /content/scaffold_runs/drjohnson20_prob_20k/point_cloud/iteration_20000/point_cloud.ply \
  --cfg_args /content/scaffold_runs/drjohnson20_prob_20k/cfg_args \
  --out /content/scaffold_runs/drjohnson20_prob_20k/scaffold_20k_probgrow_false_20prct.onnx \
  --opset 18

读取配置文件: /content/scaffold_runs/drjohnson20_prob_20k/cfg_args
已加载配置参数: feat_dim=32, appearance_dim=0, n_offsets=10, use_feat_bank=False, add_opacity_dist=False, add_cov_dist=False, add_color_dist=False
使用 MLP 路径: /content/scaffold_runs/drjohnson20_prob_20k/point_cloud/iteration_20000
/content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py:575: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`...
------camera_center
FakeTensor(..., size=(1, 3))
---
torch.Size([3371790, 3])
---
torch.Size([3371790, 3])
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
/usr/lib/python3.12/copyreg.py:99: FutureWarning

# EXPERIENCE 50%

In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

!python train.py \
    -s /content/datasets/drjohnson_50 \
    -m /content/scaffold_runs/drjohnson50_prob_20k \
    --iterations 20000 \
    --appearance_dim 0

/content/drive/MyDrive/Scaffold-GS-probgrow
0
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
2026-03-12 15:47:34.692960: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when o

In [ ]:
!python /content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py \
  --ply /content/scaffold_runs/drjohnson50_prob_20k/point_cloud/iteration_20000/point_cloud.ply \
  --cfg_args /content/scaffold_runs/drjohnson50_prob_20k/cfg_args \
  --out /content/scaffold_runs/drjohnson50_prob_20k/scaffold_20k_probgrow_true_50prct.onnx \
  --opset 18

# DATASET FICUS

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/datasets/south-building/images")
dst_root = Path("/content/datasets/south_building")
dst_images = dst_root / "images"

if dst_root.exists():
    shutil.rmtree(dst_root)

dst_images.mkdir(parents=True, exist_ok=True)

imgs = sorted(src.glob("*.JPG"))
selected = imgs[::2]   # 50%

for p in selected:
    shutil.copy2(p, dst_images / p.name)

print("Images sélectionnées:", len(selected), "/", len(imgs))

Images sélectionnées: 64 / 128


In [ ]:
!rm -f /content/datasets/south_building/database.db
!rm -rf /content/datasets/south_building/sparse
!mkdir -p /content/datasets/south_building/sparse

!QT_QPA_PLATFORM=offscreen colmap feature_extractor \
    --database_path /content/datasets/south_building/database.db \
    --image_path /content/datasets/south_building/images \
    --ImageReader.single_camera 1 \
    --SiftExtraction.use_gpu 0

!QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher \
    --database_path /content/datasets/south_building/database.db \
    --SiftMatching.use_gpu 0

!QT_QPA_PLATFORM=offscreen colmap mapper \
    --database_path /content/datasets/south_building/database.db \
    --image_path /content/datasets/south_building/images \
    --output_path /content/datasets/south_building/sparse


Feature extraction

^C

Exhaustive feature matching

F0312 18:26:45.895828 91083 cache.h:132] Check failed: max_num_elems > 0 (0 vs. 0) 
*** Check failure stack trace: ***
    @     0x7d60aab29b03  google::LogMessage::Fail()
    @     0x7d60aab319d1  google::LogMessage::SendToLog()
    @     0x7d60aab297c2  google::LogMessage::Flush()
    @     0x7d60aab2b78f  google::LogMessageFatal::~LogMessageFatal()
    @     0x584acffce76d  (unknown)
    @     0x584acffbcc9a  colmap::FeatureMatcherCache::Setup()
    @     0x584acffc92b0  colmap::ExhaustiveFeatureMatcher::Run()
    @     0x584ad00a3786  colmap::Thread::RunFunc()
    @     0x7d60a91d1253  (unknown)
    @     0x7d60a8e0dac3  (unknown)
    @     0x7d60a8e9f850  (unknown)

Loading database

Loading cameras... 1 in 0.000s
Loading matches... 0 in 0.000s
Loading images... 0 in 0.000s (connected 0)
Building correspondence graph... in 0.000s (ignored 0)

Elapsed time: 0.000 [minutes]


ERROR: failed to create sparse model


In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

!python train.py \
    -s /content/datasets/south_building \
    -m /content/scaffold_runs/south_building_false_5k \
    --iterations 5000 \
    --appearance_dim 0

# SOUTH BUILDING :

In [ ]:
!mkdir -p /content/datasets

In [ ]:
!mkdir -p /content/scaffold_runs

In [ ]:
# =========================================================
# 1) Installer COLMAP si absent
# =========================================================
!which colmap || (apt-get update -y && apt-get install -y colmap)

# =========================================================
# 2) Préparer south_building en version undistorted + PINHOLE
# =========================================================
import os
import shutil
from pathlib import Path

src_root = Path("/content/drive/MyDrive/datasets/south-building")
src_images = src_root / "images"
src_sparse0 = src_root / "sparse" / "0"
src_db = src_root / "database.db"

dst_root = Path("/content/datasets/south_building_pinhole")
undist_tmp = Path("/content/datasets/south_building_undist_tmp")

assert src_root.exists(), f"Introuvable: {src_root}"
assert src_images.exists(), f"Introuvable: {src_images}"
assert src_sparse0.exists(), f"Introuvable: {src_sparse0}"
assert src_db.exists(), f"Introuvable: {src_db}"

for f in ["cameras.txt", "images.txt", "points3D.txt"]:
    assert (src_sparse0 / f).exists(), f"Manque: {src_sparse0 / f}"

# nettoyage
for p in [dst_root, undist_tmp]:
    if p.exists():
        shutil.rmtree(p)

undist_tmp.mkdir(parents=True, exist_ok=True)

print("=== Aperçu du modèle caméra d'origine ===")
!sed -n '1,20p' /content/drive/MyDrive/datasets/south-building/sparse/0/cameras.txt

# =========================================================
# 3) Undistort avec COLMAP
#    -> ça produit un dataset exploitable par les pipelines GS
# =========================================================
!colmap image_undistorter \
    --image_path /content/drive/MyDrive/datasets/south-building/images \
    --input_path /content/drive/MyDrive/datasets/south-building/sparse/0 \
    --output_path /content/datasets/south_building_undist_tmp \
    --output_type COLMAP

# =========================================================
# 4) Organiser comme attendu par Scaffold-GS
# =========================================================
# image_undistorter écrit généralement :
#   undist_tmp/images
#   undist_tmp/sparse
# On veut :
#   dst_root/images
#   dst_root/sparse/0/{cameras.bin, images.bin, points3D.bin}

dst_sparse0 = dst_root / "sparse" / "0"
dst_sparse0.mkdir(parents=True, exist_ok=True)

# copie images undistortées
shutil.copytree(undist_tmp / "images", dst_root / "images")

# copie sparse -> sparse/0
for name in os.listdir(undist_tmp / "sparse"):
    shutil.copy2(undist_tmp / "sparse" / name, dst_sparse0 / name)

# copie database si tu veux garder une structure proche de l'ancien dataset
shutil.copy2(src_db, dst_root / "database.db")

# project.ini
(dst_sparse0 / "project.ini").write_text(
    "[project]\n"
    f"database_path={dst_root / 'database.db'}\n"
    f"image_path={dst_root / 'images'}\n"
)

print("\n=== Vérification finale ===")
for p in [
    dst_root / "images",
    dst_root / "database.db",
    dst_sparse0 / "cameras.bin",
    dst_sparse0 / "images.bin",
    dst_sparse0 / "points3D.bin",
    dst_sparse0 / "project.ini",
]:
    print(("OK   " if p.exists() else "MISS "), p)

print("\nDataset prêt :", dst_root)

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

!python train.py \
    -s /content/datasets/south_building_pinhole \
    -m /content/scaffold_runs/south_building_pinhole_test_probgrow_true \
    --iterations 5000 \
    --appearance_dim 0

/content/drive/MyDrive/Scaffold-GS-probgrow
0
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
2026-03-16 22:57:20.976700: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when o

In [ ]:
!python /content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py \
  --ply /content/scaffold_runs/south_building_pinhole_test_probgrow_true/point_cloud/iteration_5000/point_cloud.ply \
  --cfg_args /content/scaffold_runs/south_building_pinhole_test_probgrow_true/cfg_args \
  --out /content/scaffold_runs/south_building_pinhole_test_probgrow_true/south_building_5k_probgrow_true_100prct.onnx \
  --opset 18

读取配置文件: /content/scaffold_runs/south_building_pinhole_test_probgrow_true/cfg_args
已加载配置参数: feat_dim=32, appearance_dim=0, n_offsets=10, use_feat_bank=False, add_opacity_dist=False, add_cov_dist=False, add_color_dist=False
使用 MLP 路径: /content/scaffold_runs/south_building_pinhole_test_probgrow_true/point_cloud/iteration_5000
/content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py:575: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`...
------camera_center
FakeTensor(..., size=(1, 3))
---
torch.Size([1769080, 3])
---
torch.Size([1769080, 3])
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
/usr/li

# SOUTH BUILDING SPARSE VIEW

In [ ]:
# =========================================================
# Création sparse-view via COLMAP en mode headless
# =========================================================

import os
import shutil
import numpy as np
from pathlib import Path

src_root = Path("/content/datasets/south_building_pinhole")
src_images = src_root / "images"

# Force Qt en mode headless
os.environ["QT_QPA_PLATFORM"] = "offscreen"

def run_cmd(cmd):
    print("\n[CMD]", cmd)
    code = os.system(cmd)
    if code != 0:
        raise RuntimeError(f"Commande échouée avec code {code}: {cmd}")

def create_sparse_dataset(ratio, name, seed=42):
    dst_root = Path(f"/content/datasets/{name}")
    dst_images = dst_root / "images"
    dst_sparse = dst_root / "sparse"
    dst_db = dst_root / "database.db"

    if dst_root.exists():
        shutil.rmtree(dst_root)

    dst_images.mkdir(parents=True, exist_ok=True)
    dst_sparse.mkdir(parents=True, exist_ok=True)

    print(f"\n=== {name} ({int(ratio*100)}%) ===")

    # --- sélectionner sous-ensemble d’images ---
    all_images = sorted([
        f for f in os.listdir(src_images)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    rng = np.random.default_rng(seed)
    n_keep = max(2, int(len(all_images) * ratio))
    selected = sorted(rng.choice(all_images, size=n_keep, replace=False).tolist())

    print(f"Images gardées : {len(selected)} / {len(all_images)}")

    # --- copier images retenues ---
    for img in selected:
        shutil.copy2(src_images / img, dst_images / img)

    # --- liste explicite des images ---
    image_list_path = dst_root / "selected_images.txt"
    with open(image_list_path, "w") as f:
        for img in selected:
            f.write(img + "\n")

    # --- base COLMAP neuve ---
    if dst_db.exists():
        dst_db.unlink()

    # --- feature extraction ---
    run_cmd(
        f'QT_QPA_PLATFORM=offscreen colmap feature_extractor '
        f'--database_path "{dst_db}" '
        f'--image_path "{dst_images}" '
        f'--image_list_path "{image_list_path}" '
        f'--ImageReader.single_camera 1 '
        f'--ImageReader.camera_model PINHOLE '
        f'--SiftExtraction.use_gpu 0'
    )

    # --- matching ---
    run_cmd(
        f'QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher '
        f'--database_path "{dst_db}" '
        f'--SiftMatching.use_gpu 0'
    )

    # --- mapping ---
    run_cmd(
        f'QT_QPA_PLATFORM=offscreen colmap mapper '
        f'--database_path "{dst_db}" '
        f'--image_path "{dst_images}" '
        f'--output_path "{dst_sparse}"'
    )

    sparse0 = dst_sparse / "0"
    if not sparse0.exists():
        raise RuntimeError(f"COLMAP n'a pas créé {sparse0}. Reconstruction échouée.")

    # --- project.ini ---
    (sparse0 / "project.ini").write_text(
        "[project]\n"
        f"database_path={dst_db}\n"
        f"image_path={dst_images}\n"
    )

    print("\nVérification finale :")
    for p in [
        dst_images,
        dst_db,
        sparse0 / "cameras.bin",
        sparse0 / "images.bin",
        sparse0 / "points3D.bin",
        sparse0 / "project.ini",
    ]:
        print(("OK   " if p.exists() else "MISS "), p)

    print(f"\nDataset prêt : {dst_root}")


# =========================================================
# Génération
# =========================================================

create_sparse_dataset(0.5, "south_building_50", seed=42)
create_sparse_dataset(0.2, "south_building_20", seed=42)


=== south_building_50 (50%) ===
Images gardées : 64 / 128

[CMD] QT_QPA_PLATFORM=offscreen colmap feature_extractor --database_path "/content/datasets/south_building_50/database.db" --image_path "/content/datasets/south_building_50/images" --image_list_path "/content/datasets/south_building_50/selected_images.txt" --ImageReader.single_camera 1 --ImageReader.camera_model PINHOLE --SiftExtraction.use_gpu 0

[CMD] QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher --database_path "/content/datasets/south_building_50/database.db" --SiftMatching.use_gpu 0

[CMD] QT_QPA_PLATFORM=offscreen colmap mapper --database_path "/content/datasets/south_building_50/database.db" --image_path "/content/datasets/south_building_50/images" --output_path "/content/datasets/south_building_50/sparse"

Vérification finale :
OK    /content/datasets/south_building_50/images
OK    /content/datasets/south_building_50/database.db
OK    /content/datasets/south_building_50/sparse/0/cameras.bin
OK    /content/dataset

In [ ]:
%cd /content/drive/MyDrive/Scaffold-GS-probgrow

!python train.py \
    -s /content/datasets/south_building_20 \
    -m /content/scaffold_runs/south_building_20_test_probgrow_true__hyper_expl_params \
    --iterations 5000 \
    --appearance_dim 0

/content/drive/MyDrive/Scaffold-GS-probgrow
0
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
2026-03-18 23:53:27.684851: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when o

In [ ]:
!python /content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py \
  --ply /content/scaffold_runs/south_building_20_test_probgrow_true__hyper_expl_params/point_cloud/iteration_5000/point_cloud.ply \
  --cfg_args /content/scaffold_runs/south_building_20_test_probgrow_true__hyper_expl_params/cfg_args \
  --out /content/scaffold_runs/south_building_20_test_probgrow_true__hyper_expl_params/south_building_5k_probgrow_true_20prct_hyper_expl_params.onnx \
  --opset 18

读取配置文件: /content/scaffold_runs/south_building_20_test_probgrow_true__hyper_expl_params/cfg_args
已加载配置参数: feat_dim=32, appearance_dim=0, n_offsets=10, use_feat_bank=False, add_opacity_dist=False, add_cov_dist=False, add_color_dist=False
使用 MLP 路径: /content/scaffold_runs/south_building_20_test_probgrow_true__hyper_expl_params/point_cloud/iteration_5000
/content/visionary/onnx-export/ONNXExample-scaffold/onnx_template.py:575: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`...
------camera_center
FakeTensor(..., size=(1, 3))
---
torch.Size([9958710, 3])
---
torch.Size([9958710, 3])
[torch.onnx] Obtain model graph for `GaussianSetModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] 

Maintenant :

Faire les expériences 50% et 20% avec south building (5k et 20k).

Quels paramètres est-ce qu'on pourrait changer ? Tester des changements de paramètres sur south_building. On voit que ça marche bien mieux sur south_building que sur drjohnson, c'es tplus petit et plus adapté.